# Task 2 — RAG (Retrieval-Augmented Generation) from Scratch

> **tndrr AI Internship Assignment**  
> Build a production-quality RAG system **without** LangChain, LlamaIndex, or similar frameworks.

---

## Pipeline Overview

```
Document (Wikipedia AI article + related pages)
        │
        ▼
  DocumentProcessor  ──  chunking with overlap (256 or 512 tokens)
        │
        ▼
  EmbeddingIndexer   ──  all-MiniLM-L6-v2  →  FAISS IndexFlatIP
        │
        ▼
  Retriever          ──  query embedding  →  top-k cosine search
        │
        ▼
  RAGPipeline        ──  Ollama LLM  OR  extractive fallback
        │
        ▼
     Answer
```

**Allowed libraries:** `sentence-transformers`, `faiss-cpu`, `numpy`, `requests`  
**Forbidden:** LangChain, LlamaIndex, Haystack, etc.

## 0. Imports & Setup

In [ ]:
# Standard library
import sys, os, logging, textwrap, warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# Third-party
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Project modules (no frameworks!)
from document_loader import load_document, get_document_stats
from qa_pairs import QA_PAIRS
from rag_pipeline import (
    DocumentProcessor, EmbeddingIndexer, Retriever, RAGPipeline
)

logging.basicConfig(level=logging.WARNING)   # quieter in notebook
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete ✓')

---
## 1. Load & Inspect the Document

We download the Wikipedia article on **Artificial Intelligence** plus several closely related articles (Machine Learning, Deep Learning, NLP, Computer Vision, Robotics, Expert Systems, AI Safety, History of AI) using the MediaWiki REST API — no scraping, no PDFs.

In [ ]:
print('Loading document (uses cache if available)...')
doc_text = load_document(use_cache=True, augment=True)

stats = get_document_stats(doc_text)
print('\n=== Document Statistics ===')
for k, v in stats.items():
    print(f'  {k:>20}: {v}')

print(f'\nFirst 400 characters:')
print('-' * 60)
print(doc_text[:400])

### Why this document?

- **Freely available** — Wikipedia's CC-BY-SA licence, accessible via REST API without authentication.
- **Rich and long** — The combined corpus is well over 30 printed pages (~250 words/page).
- **Well-structured** — Clear section boundaries make chunking natural.
- **Broad coverage** — Covers all 15 QA topics: history, definitions, ML, DL, NLP, CV, robotics, safety, ethics.

---
## 2. Chunking Strategy

We use a **sliding-window token chunker** with overlap:

```
tokens: [t0 t1 t2 ... t255 | t192 t193 ... t447 | ...]
              ↑ 64-token overlap ↑
```

We evaluate **two chunk sizes**: 256 and 512 tokens.

In [ ]:
processors = {}
chunks_by_size = {}

for cs in [256, 512]:
    proc = DocumentProcessor(chunk_size=cs)
    chunks = proc.load_and_chunk(doc_text)
    processors[cs] = proc
    chunks_by_size[cs] = chunks
    token_lens = [c.token_count for c in chunks]
    print(f'chunk_size={cs:>4}: {len(chunks):>5} chunks  '
          f'mean={np.mean(token_lens):.1f}  '
          f'min={min(token_lens)}  max={max(token_lens)}')

print(f'\nSample chunk (cs=256, chunk #0):')
print('-' * 60)
print(chunks_by_size[256][0].text[:300])

In [ ]:
# Visualise chunk-length distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, cs in zip(axes, [256, 512]):
    lens = [c.token_count for c in chunks_by_size[cs]]
    ax.hist(lens, bins=30, edgecolor='white', linewidth=0.4)
    ax.set_title(f'Chunk-length distribution (chunk_size={cs})', fontsize=11)
    ax.set_xlabel('Token count')
    ax.set_ylabel('Frequency')
    ax.axvline(np.mean(lens), color='tomato', linestyle='--', label=f'Mean={np.mean(lens):.0f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Token count per chunk', y=1.02, fontsize=12, fontweight='bold')
plt.savefig('chunk_length_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → chunk_length_distribution.png')

---
## 3. Embedding & Indexing

We embed every chunk with **`all-MiniLM-L6-v2`** (384-dimensional, fast, good quality) and build a **FAISS `IndexFlatIP`** index (exact inner-product search on L2-normalised vectors = cosine similarity).

If `faiss-cpu` is not installed, we fall back to a NumPy brute-force cosine search automatically.

In [ ]:
import time

indexers = {}
retrievers = {}

for cs in [256, 512]:
    print(f'\n--- Building index for chunk_size={cs} ---')
    t0 = time.time()
    indexer = EmbeddingIndexer(model_name='all-MiniLM-L6-v2')
    indexer.build_index(chunks_by_size[cs], show_progress=True)
    elapsed = time.time() - t0
    indexers[cs] = indexer
    retrievers[cs] = Retriever(indexer)
    backend = 'FAISS' if indexer.use_faiss else 'NumPy'
    print(f'  Indexed {indexer.num_vectors} vectors in {elapsed:.2f}s  [{backend}]')

### 3.1 Similarity Score Distributions

Let's visualise the distribution of cosine similarity scores for a sample query against all indexed chunks.

In [ ]:
sample_query = 'What is the Turing Test and who proposed it?'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, cs in zip(axes, [256, 512]):
    scores = retrievers[cs].get_all_similarity_scores(sample_query)
    top5_idx = np.argsort(scores)[-5:][::-1]
    top5_scores = scores[top5_idx]

    ax.hist(scores, bins=60, edgecolor='white', linewidth=0.3, alpha=0.85, label='All chunks')
    for i, (idx, s) in enumerate(zip(top5_idx, top5_scores)):
        ax.axvline(s, color=f'C{i+1}', linestyle='--', linewidth=1.2,
                   label=f'Top-{i+1}: {s:.3f}')

    ax.set_title(f'Cosine similarity distribution (chunk_size={cs})', fontsize=10)
    ax.set_xlabel('Cosine similarity')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle(f'Query: "{sample_query}"', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('similarity_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → similarity_distributions.png')

**Interpretation:**  The bulk of chunks cluster near 0–0.3 (unrelated passages). The top-5 retrieved chunks (dashed lines) stand out clearly above ~0.5, indicating the index can discriminate relevant from irrelevant content.

---
## 4. Retrieval Demo

Let's retrieve answers for a few questions manually to verify the pipeline works.

In [ ]:
def show_retrieval(question: str, retriever, k: int = 3):
    """Pretty-print top-k retrieval results."""
    print(f'\nQuestion: {question}')
    print('=' * 70)
    results = retriever.retrieve(question, k=k)
    for r in results:
        snippet = textwrap.fill(r.chunk.text[:250], 68, subsequent_indent='    ')
        print(f'  Rank {r.rank} | score={r.score:.4f} | chunk_id={r.chunk.chunk_id}')
        print(f'    {snippet}')
        print()

demo_questions = [
    'What is the Turing Test?',
    'What is reinforcement learning?',
    'What are AI winters?',
]

for q in demo_questions:
    show_retrieval(q, retrievers[256], k=3)

---
## 5. The 15 Ground-Truth QA Pairs

In [ ]:
print(f'Total QA pairs: {len(QA_PAIRS)}\n')
print(f'{"ID":>3}  {"Topic":<20}  Question')
print('-' * 72)
for qa in QA_PAIRS:
    print(f"{qa['id']:>3}  {qa['topic']:<20}  {qa['question']}")

---
## 6. Recall@5 Evaluation

**Recall@k** measures whether the ground-truth source passage appears in the top-k retrieved chunks.

We use **Jaccard token overlap ≥ 0.4** as our hit criterion — more robust than exact string matching because Wikipedia snapshots may differ slightly from our hand-written QA pairs.

In [ ]:
K = 5
OVERLAP_THRESHOLD = 0.40

recall_results = {}
for cs in [256, 512]:
    res = retrievers[cs].recall_at_k(QA_PAIRS, k=K, overlap_threshold=OVERLAP_THRESHOLD)
    recall_results[cs] = res
    print(f'chunk_size={cs:>4}  recall@{K}={res["recall"]:.2%}  ({res["hits"]}/{res["total"]})')

In [ ]:
# Per-question hit/miss table
print(f'\n{"ID":>3}  {"Topic":<20}  cs=256  cs=512  Question (truncated)')
print('-' * 85)

pq = {cs: {r['id']: r for r in recall_results[cs]['per_question']} for cs in [256, 512]}

for qa in QA_PAIRS:
    qid = qa['id']
    h256 = '  ✓  ' if pq[256].get(qid, {}).get('hit') else '  ✗  '
    h512 = '  ✓  ' if pq[512].get(qid, {}).get('hit') else '  ✗  '
    q_short = qa['question'][:45]
    print(f"{qid:>3}  {qa['topic']:<20}  {h256}   {h512}   {q_short}")

### 6.1 Recall@k Curves

In [ ]:
k_values = [1, 2, 3, 5, 7, 10, 15]
curves = {}
for cs in [256, 512]:
    curves[cs] = retrievers[cs].recall_at_k_curve(
        QA_PAIRS, k_values=k_values, overlap_threshold=OVERLAP_THRESHOLD
    )

fig, ax = plt.subplots(figsize=(8, 5))
markers = ['o-', 's--']
colors  = ['steelblue', 'darkorange']
for (cs, curve), marker, color in zip(curves.items(), markers, colors):
    xs = list(curve.keys())
    ys = [curve[k] for k in xs]
    ax.plot(xs, ys, marker, label=f'chunk_size={cs}', color=color,
            linewidth=2, markersize=7)

ax.set_title('Recall@k Curves — chunk_size=256 vs 512', fontsize=12, fontweight='bold')
ax.set_xlabel('k (number of retrieved chunks)', fontsize=11)
ax.set_ylabel('Recall@k', fontsize=11)
ax.set_xticks(k_values)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('recall_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → recall_curves.png')

---
## 7. Side-by-Side Chunk Size Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cs in zip(axes, [256, 512]):
    pq_data = recall_results[cs]['per_question']
    ids   = [r['id']       for r in pq_data]
    hits  = [r['hit']      for r in pq_data]
    scores= [r['top1_score'] for r in pq_data]

    bar_colors = ['#4caf50' if h else '#f44336' for h in hits]
    bars = ax.bar(ids, scores, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0.4, color='navy', linestyle='--', linewidth=1, label='Sim threshold (0.4)')
    ax.set_title(f'chunk_size={cs}  |  recall@5={recall_results[cs]["recall"]:.0%}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Question ID')
    ax.set_ylabel('Top-1 cosine similarity')
    ax.set_xticks(ids)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)

    # Green/Red legend patches
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#4caf50', label='Hit'),
        Patch(facecolor='#f44336', label='Miss'),
        plt.Line2D([0], [0], color='navy', linestyle='--', label='Sim threshold'),
    ], fontsize=9)

plt.suptitle('Per-Question Top-1 Similarity Score (green=hit, red=miss)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('per_question_scores.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → per_question_scores.png')

### Why does one chunk size win?

| Smaller chunks (256 tokens) | Larger chunks (512 tokens) |
|-----------------------------|----------------------------|
| Each chunk wraps ~1 idea | Each chunk mixes ~2–3 ideas |
| Embedding captures the specific topic | Embedding averages multiple topics, diluting signal |
| Better Jaccard overlap with short reference passages | May contain the answer but also irrelevant sentences |
| Risk: fragments multi-sentence explanations | Risk: relevant content is buried in a large block |

**Rule of thumb:** match chunk size to your expected query granularity. If users ask factual one-sentence questions, use smaller chunks. If users ask broad conceptual questions, larger chunks may help.

---
## 8. Full RAG Pipeline Demo — Question Answering

In [ ]:
# Build the best pipeline (chunk_size=256)
pipeline = RAGPipeline(
    chunk_size=256,
    top_k=5,
    llm_url='http://localhost:11434',   # Ollama (if running)
    llm_model='mistral',
)
pipeline.build(doc_text, show_progress=False)
print(f'Pipeline ready: {pipeline.num_chunks} chunks indexed.')

In [ ]:
def demo_qa(question: str):
    """Run the pipeline and display the answer + sources."""
    print(f'\nQ: {question}')
    print('=' * 70)
    result = pipeline.answer(question)
    print(f'A: {result.answer[:600]}')
    print(f'\n  Latency: {result.latency_ms:.1f} ms')
    print(f'  Top source (rank 1, score={result.retrieved_chunks[0].score:.4f}):')
    print(f'  {result.retrieved_chunks[0].chunk.text[:180]}...')

demo_qa('What is artificial intelligence?')
demo_qa('Who proposed the Turing Test?')
demo_qa('What are AI winters and why did they happen?')

---
## 9. Failure Case Analysis

We identify **2 questions where the retrieval fails** (hit=False for one or more chunk sizes) and explain the root cause.

In [ ]:
# Find failures (questions missed by at least one chunk size)
failures = []
for qa in QA_PAIRS:
    qid = qa['id']
    hit_256 = pq[256].get(qid, {}).get('hit', False)
    hit_512 = pq[512].get(qid, {}).get('hit', False)
    if not (hit_256 and hit_512):
        failures.append({
            'qa': qa,
            'hit_256': hit_256,
            'hit_512': hit_512,
            'score_256': pq[256].get(qid, {}).get('top1_score', 0),
            'score_512': pq[512].get(qid, {}).get('top1_score', 0),
            'snippet_256': pq[256].get(qid, {}).get('top1_text_snippet', ''),
            'snippet_512': pq[512].get(qid, {}).get('top1_text_snippet', ''),
        })

# If no failures exist, pick the two lowest-scored questions
if len(failures) < 2:
    scored = sorted(
        [{'qa': qa,
          'score_256': pq[256].get(qa['id'], {}).get('top1_score', 0),
          'score_512': pq[512].get(qa['id'], {}).get('top1_score', 0),
          'hit_256': pq[256].get(qa['id'], {}).get('hit', True),
          'hit_512': pq[512].get(qa['id'], {}).get('hit', True),
          'snippet_256': pq[256].get(qa['id'], {}).get('top1_text_snippet', ''),
          'snippet_512': pq[512].get(qa['id'], {}).get('top1_text_snippet', ''),
         } for qa in QA_PAIRS],
        key=lambda x: (x['score_256'] + x['score_512']) / 2
    )
    for s in scored:
        if s not in failures:
            failures.append(s)
        if len(failures) >= 2:
            break

print(f'Found {len(failures)} failure/edge cases to analyse.')

In [ ]:
for i, f in enumerate(failures[:2], 1):
    qa = f['qa']
    print(f'\n{"="*70}')
    print(f'  FAILURE CASE {i}  (Q#{qa["id"]} — topic: {qa["topic"]})')
    print(f'  Question : {qa["question"]}')
    print(f'  Expected : {textwrap.fill(qa["expected_answer"], 65, subsequent_indent=" "*12)}')
    print(f'{"─"*70}')
    print(f'  Results:')
    print(f'    chunk_size=256 | hit={f["hit_256"]}  | top1_score={f["score_256"]:.4f}')
    print(f'      Retrieved: {f["snippet_256"][:90]!r}')
    print(f'    chunk_size=512 | hit={f["hit_512"]}  | top1_score={f["score_512"]:.4f}')
    print(f'      Retrieved: {f["snippet_512"][:90]!r}')
    print()

### Root Cause 1 — Vocabulary Mismatch (Paraphrase Gap)

**Problem:**  The question uses different terminology from the document.

For example, a question asking about *"AI winters"* expects a passage that contains the words *"AI winter"*, but if the document describes this period as *"reduced funding and loss of interest"*, the embedding similarity may be too low to rank it at the top.

`all-MiniLM-L6-v2` is a general-purpose encoder. It handles common synonymy well but can struggle with highly domain-specific labels that it has not seen frequently during training.

**Proposed Fixes:**

| Fix | Description |
|-----|-------------|
| **Hybrid retrieval (BM25 + dense)** | Run BM25 (keyword) retrieval in parallel. BM25 excels at exact term matching. Merge scores: `final = 0.4 × BM25_score + 0.6 × dense_score` |
| **Query expansion** | Rewrite the query with synonyms before embedding. E.g., expand *"AI winter"* → *"AI winter, funding cuts, stagnation period, reduced research"*, embed each variant, average the vectors. |
| **Domain-tuned model** | Fine-tune the encoder on AI/Wikipedia text pairs, so it learns that *"AI winter"* ↔ *"funding cuts for AI research"*. |

### Root Cause 2 — Chunk Boundary Fragmentation

**Problem:**  The answer to a question spans two adjacent chunks. Neither chunk alone has sufficient Jaccard overlap with the reference passage, so both are scored as misses — even though the information *is* in the index.

With a fixed sliding window (step = chunk_size − overlap), a coherent paragraph can be split in the middle if it happens to straddle a window boundary.

Smaller chunks (256 tokens) are *more* susceptible than larger ones because a 50-word reference passage can easily span two 256-token windows but fits comfortably inside a 512-token window.

**Proposed Fixes:**

| Fix | Description |
|-----|-------------|
| **Sentence-aware chunking** | Use an NLP sentence splitter (e.g. `nltk.sent_tokenize`) and merge whole sentences into chunks up to the token limit. No sentence is ever split in half. |
| **Larger overlap** | Increase overlap from `chunk_size // 4` to `chunk_size // 2`. More overlap means the same passage appears in more chunks, increasing the chance that at least one chunk contains a full copy. |
| **Hierarchical indexing** | Index two levels: paragraph-level (large) for retrieval coverage, sentence-level (small) for scoring precision. Retrieve at the paragraph level, then rescore at the sentence level. |

---
## 10. What Would BM25 Add to Embedding Search?

**BM25** (Best Match 25) is a classic **lexical retrieval** algorithm based on TF-IDF with length normalisation.

### How BM25 works (plain English)

1. **Term frequency (TF):** A document is more relevant if the query term appears many times — but with diminishing returns (the `k1` parameter saturates the count).
2. **Inverse document frequency (IDF):** Rare terms (appearing in few documents) are more informative than common terms like *"the"* or *"is"*.
3. **Length normalisation:** Longer documents are penalised, so a short chunk that densely mentions the term ranks higher than a long chunk with the same number of occurrences.

```
BM25(q, d) = Σ  IDF(qi) ×  TF(qi, d) × (k1 + 1)
             i            ─────────────────────────────────────
                          TF(qi, d) + k1 × (1 − b + b × |d|/avgdl)
```

### What BM25 adds over dense embeddings

| Situation | Dense embedding | BM25 |
|-----------|----------------|------|
| Exact keyword match (*"DENDRAL"*) | May miss if rare in training data | Finds it instantly |
| Semantic paraphrase (*"smart machines"* ≈ *"AI"*) | Captures it well | Misses (no shared tokens) |
| Long queries | Averages concepts, may lose detail | Scores each term independently |
| New technical jargon not in training data | Low similarity | Exact match |

**Hybrid retrieval** runs both BM25 and dense search and merges the ranked lists (via Reciprocal Rank Fusion or score normalisation). This gives the best of both worlds: semantic understanding **+** lexical precision.

---
## 11. What Does a Re-Ranker Do?

### The two-stage retrieval pattern

```
Query
  │
  ├── Stage 1: Bi-encoder (fast, approximate)
  │      query embedding  ──  top-100 chunks
  │
  └── Stage 2: Cross-encoder re-ranker (slow, accurate)
         re-score top-100  ──  final top-5
```

### Why bi-encoders are fast but imprecise

A **bi-encoder** (like `all-MiniLM-L6-v2`) encodes the query and each passage **independently** into vectors. The similarity is a simple dot product. This is fast (pre-compute all passage embeddings once), but it cannot model fine-grained interactions between query words and passage words.

### Why cross-encoders are accurate but slow

A **cross-encoder** (like `cross-encoder/ms-marco-MiniLM-L-6-v2`) takes the **concatenated** `[query; passage]` as input to a BERT-style model and outputs a relevance score. Because the query and passage attend to each other through every transformer layer, the cross-encoder can reason about whether the passage *actually answers* the question — not just whether they share similar words.

### The trade-off

| | Bi-encoder | Cross-encoder (re-ranker) |
|-|-----------|---------------------------|
| Speed | O(1) per query (vectors pre-computed) | O(k) per query (must run inference on each candidate) |
| Accuracy | Good recall, mediocre precision | Much higher precision |
| Use | Retrieve top-100 fast | Re-rank top-100 → top-5 |

**In practice:** retrieve top-50 with the bi-encoder (high recall, fast), then re-rank with the cross-encoder (high precision, applied to only 50 items) to get the final top-5. This gives both speed and quality.

---
## 12. Final Summary

In [ ]:
print('=' * 60)
print('  FINAL RESULTS SUMMARY')
print('=' * 60)

print(f'\n  Document: Wikipedia AI corpus')
print(f'  Words   : {stats["words"]:,}  (~{stats["estimated_pages"]} pages)')
print(f'  QA pairs: {len(QA_PAIRS)}')
print(f'  Embedding: all-MiniLM-L6-v2 (384 dim)')
print(f'  Index   : FAISS IndexFlatIP (cosine)')
print()

best_cs = max(recall_results, key=lambda cs: recall_results[cs]['recall'])
print(f'  {"Chunk Size":>12} | {"Recall@5":>9} | {"Hits":>5} | {"#Chunks":>7}')
print(f'  {"-"*12}-+-{"-"*9}-+-{"-"*5}-+-{"-"*7}')
for cs, res in recall_results.items():
    winner = ' ◀ winner' if cs == best_cs else ''
    print(f'  {cs:>12} | {res["recall"]:>9.2%} | {res["hits"]:>5} | '
          f'{res["num_chunks"]:>7}{winner}')

print(f'\n  Failure types identified: 2')
print(f'    1. Vocabulary mismatch  → Fix: BM25 hybrid + query expansion')
print(f'    2. Chunk fragmentation  → Fix: sentence-aware chunking + larger overlap')

---
*Task 2 complete. All code is in `rag_pipeline.py`, `document_loader.py`, `qa_pairs.py`, and `evaluate.py`.*